[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/54_planning_solution.ipynb)

#  Solution: Planning & Self-Correction Loop

Reference: ReAct (Yao et al.), Reflexion (Shinn et al. 2023), Tree-of-Thought (Yao et al. 2023), ReST-EM (Singh et al. 2024).

```text
1. Generate plan via planner_fn
2. Execute each step sequentially
3. If step fails: retry up to max_retries, then re-plan remaining
4. Track trace: list of {step, result, success, retry_count}
5. Return final result + trace
```

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  SOLUTION

def planning_execute_loop(task, planner_fn, executor_fn,
                         max_retries=3, verifier_fn=None):
    """Plan-then-execute with self-correction.

    Inspired by:
    - ReAct (Yao et al. 2023): interleaved reasoning + acting
    - Reflexion (Shinn et al. 2023): verbal self-reflection on failures
    - Tree-of-Thought (Yao et al. 2023): structured backtracking over reasoning paths
    - ReST-EM (Singh et al. 2024): reinforcement self-training with execution feedback

    Core idea: don't just fail — analyze WHY you failed and try differently.
    This is the essence of agent self-improvement.
    """
    context = {"task": task, "replan_count": 0}
    trace = []

    plan = planner_fn(task, context)
    if not plan:
        return {"result": None, "success": False}, trace

    step_idx = 0
    while step_idx < len(plan):
        step = plan[step_idx]
        retry_count = 0

        while retry_count < max_retries:
            exec_result = executor_fn(step, context)
            success = exec_result.get("success", False)
            trace.append({
                "step": step,
                "result": exec_result,
                "success": success,
                "retry": retry_count,
            })

            if success:
                break
            retry_count += 1

        if retry_count >= max_retries and not success:
            # Re-plan remaining steps
            context["error"] = f"Step '{step}' failed after {max_retries} retries"
            context["replan_count"] += 1
            remaining_plan = planner_fn(task, context)
            plan[step_idx+1:] = remaining_plan
            # If re-plan returns same plan, give up to avoid infinite loop
            if remaining_plan and remaining_plan[0] == step:
                return {"result": f"Failed: {step}", "success": False}, trace

        step_idx += 1

    final = trace[-1]["result"] if trace else {"result": None, "success": False}
    return final, trace

In [ ]:
#  Verify
attempts = {"step_b": 0}
def planner(task, ctx):
    if ctx.get("replan_count", 0) > 0:
        return ["step_x", "step_y"]
    return ["step_a", "step_b", "step_c"]
def executor(step, ctx):
    if step == "step_b":
        attempts["step_b"] += 1
        if attempts["step_b"] < 3:
            return {"result": "error", "success": False}
    return {"result": f"done_{step}", "success": True}
result, trace = planning_execute_loop("task", planner, executor, max_retries=3)
print(f"Result: {result['success']}, attempts: {attempts}, trace: {len(trace)} steps")

In [ ]:
from torch_judge import check
check("planning")